# Learning Airflow, Dagster, and Prefect

All three are Python workflow orchestrators: you write a pipeline as code, they schedule/run/retry/observe it. They differ in mental model and how much infrastructure they need just to run one pipeline locally:

| | Core unit | Local dev model | Needs a metadata DB even for one local run? |
|---|---|---|---|
| **Prefect** | `@flow` / `@task` functions | call the flow function like normal Python | no (spins up an ephemeral in-memory API automatically) |
| **Dagster** | `@asset` (the *data* a step produces, not the step itself) | `materialize([...])` in-process | no (ephemeral run, in a temp dir) |
| **Airflow** | `DAG` of `@task`s | DAG file must live in a `dags/` folder, gets picked up by a metadata DB | yes, always |

Each section below installs one tool in isolation and runs the *same* tiny pipeline (extract 5 numbers → square them → sum them) so the differences in ceremony are easy to compare directly.

Run each tool's section as its own kernel/environment if you hit dependency conflicts — Airflow 3 and Prefect 3 currently pin incompatible `fastapi` versions, which is realistic: **these tools are normally run in separate environments/services, not imported into the same process.**

## 1. Prefect — plain Python functions, decorated

`@task` marks a unit of work, `@flow` marks the pipeline. Calling the flow just calls the function — Prefect wraps it with retries, logging, and state tracking, and spins up a lightweight local API on first use so you don't need a running server.

In [ ]:
%pip install prefect

In [ ]:
from prefect import flow, task


@task
def extract():
    return [1, 2, 3, 4, 5]


@task
def transform(numbers):
    return [n * n for n in numbers]


@task
def load(numbers):
    total = sum(numbers)
    print(f"Loaded total: {total}")
    return total


@flow(name="square-and-sum")
def etl_flow():
    raw = extract()
    squared = transform(raw)
    return load(squared)


result = etl_flow()
print("Flow result:", result)

## 2. Dagster — assets, not tasks

Dagster's unit is the **asset**: the data a step produces (`squared_numbers`), not the act of producing it. A downstream asset simply names an upstream asset as a function parameter and Dagster wires the dependency graph for you. `materialize()` runs the whole graph in-process, in a temp directory, with no separate service required.

In [ ]:
%pip install dagster

In [ ]:
from dagster import asset, materialize


@asset
def raw_numbers():
    return [1, 2, 3, 4, 5]


@asset
def squared_numbers(raw_numbers):
    return [n * n for n in raw_numbers]


@asset
def total(squared_numbers):
    return sum(squared_numbers)


result = materialize([raw_numbers, squared_numbers, total])
print("success:", result.success)
print("total value:", result.output_for_node("total"))

## 3. Airflow — DAGs are files, not cells

Airflow is the odd one out: even `dag.test()`, its local-testing entry point, requires the DAG to be a real file inside a `dags/` folder and registered in Airflow's metadata database first (`airflow db migrate`, then `airflow dags reserialize`). You cannot just define a DAG inline in a notebook cell and call it — this friction is a real, structural difference from Prefect/Dagster, not an artifact of this demo.

So this section writes the DAG to a file on disk, runs the two setup commands as subprocesses, then imports and tests it — the minimum ceremony Airflow 3 actually requires for a local run.

In [ ]:
%pip install apache-airflow

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

AIRFLOW_HOME = Path("./airflow_home").resolve()
DAGS_FOLDER = AIRFLOW_HOME / "dags"
DAGS_FOLDER.mkdir(parents=True, exist_ok=True)
os.environ["AIRFLOW_HOME"] = str(AIRFLOW_HOME)

dag_file = DAGS_FOLDER / "square_and_sum.py"
dag_file.write_text('''
from datetime import datetime

from airflow.sdk import DAG, task

with DAG(dag_id="square_and_sum", start_date=datetime(2026, 1, 1), schedule=None) as dag:

    @task
    def extract():
        return [1, 2, 3, 4, 5]

    @task
    def transform(numbers):
        return [n * n for n in numbers]

    @task
    def load(numbers):
        total = sum(numbers)
        print(f"Loaded total: {total}")
        return total

    load(transform(extract()))
''')

In [ ]:
# One-time setup: create the metadata DB, then register the DAG file in it
subprocess.run([sys.executable, "-m", "airflow", "db", "migrate"], check=True)
subprocess.run([sys.executable, "-m", "airflow", "dags", "reserialize"], check=True)

In [ ]:
sys.path.insert(0, str(DAGS_FOLDER))
import square_and_sum  # the DAG module we just wrote to disk

square_and_sum.dag.test()

## Takeaways

- **Prefect**: lowest ceremony, plain functions, dynamic pipelines (loops/conditionals work naturally since it's just Python), good default choice for small-to-mid teams.
- **Dagster**: thinks in data assets rather than tasks, so lineage/freshness/data-quality checks are first-class; best when the data products themselves (tables, files, ML models) are what you want to reason about.
- **Airflow**: the incumbent, huge ecosystem of providers/operators for every external system, but the DAG-file + metadata-DB model shown above is real overhead for local iteration; it earns its keep at scale with a real scheduler and many pipelines, not for a single quick script.